In [ ]:
# read it in to inspect it
with open('../data/dialogues/train.txt', 'r', encoding='utf-8') as f:
    text = f.read()
text = text.lower()
with open('../data/dialogues/test.txt', 'r', encoding='utf-8') as f:
    test_text = f.read()
test_text = test_text.lower()

In [36]:
print("length of dataset in characters: ", len(text))
print("length of test dataset in characters: ", len(test_text))

length of dataset in characters:  5085870
length of test dataset in characters:  459230


In [37]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)
# 计算每个字符的频率
from collections import Counter
char_counts = Counter(text)
# 打印每个字符及其频率
for char, count in char_counts.items():
    print(f"'{char}': {count}")


 ',.?abcdefghijklmnopqrstuvwxyz
32
's': 222875
'a': 297446
'y': 132242
',': 46280
' ': 882799
'j': 8475
'i': 282465
'm': 104112
'h': 203834
'o': 350567
'w': 95237
'b': 57622
'u': 143433
't': 360524
'g': 82787
'n': 240667
'f': 68315
'r': 211614
'e': 454236
'd': 129437
'?': 33103
'
': 98280
'k': 52704
'p': 63445
'l': 166864
'.': 106048
'x': 6820
''': 42981
'c': 93906
'v': 41937
'z': 2117
'q': 2698


In [38]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[13, 14, 14, 1, 25, 13, 10, 23, 10]
hii there


In [40]:
import torch # we use PyTorch: https://pytorch.org
device = 'cuda:4' if torch.cuda.is_available() else 'cpu'
data = torch.tensor(encode(text), dtype=torch.long).to(device)
data_test = torch.tensor(encode(test_text), dtype=torch.long).to(device)
print(data.shape, data.dtype)
print(data_test.shape, data_test.dtype)

torch.Size([5085870]) torch.int64
torch.Size([459230]) torch.int64


In [41]:
# n = int(0.9*len(data)) # first 90% will be train, rest val
# train_data = data[:n]
# val_data = data[n:]
train_data, val_data = data, data_test

In [43]:
block_size = 8
train_data[:block_size+1]

tensor([24,  6, 30,  3,  1, 15, 14, 18,  3], device='cuda:4')

In [44]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([24], device='cuda:4') the target: 6
when input is tensor([24,  6], device='cuda:4') the target: 30
when input is tensor([24,  6, 30], device='cuda:4') the target: 3
when input is tensor([24,  6, 30,  3], device='cuda:4') the target: 1
when input is tensor([24,  6, 30,  3,  1], device='cuda:4') the target: 15
when input is tensor([24,  6, 30,  3,  1, 15], device='cuda:4') the target: 14
when input is tensor([24,  6, 30,  3,  1, 15, 14], device='cuda:4') the target: 18
when input is tensor([24,  6, 30,  3,  1, 15, 14, 18], device='cuda:4') the target: 3


In [45]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[25, 20,  1,  9, 20,  1, 28, 13],
        [14, 24, 25, 14, 19,  1,  9, 14],
        [10,  1, 25, 13, 10, 14, 23,  1],
        [13, 10,  1, 21, 23, 20,  7, 17]], device='cuda:4')
targets:
torch.Size([4, 8])
tensor([[20,  1,  9, 20,  1, 28, 13, 10],
        [24, 25, 14, 19,  1,  9, 14,  9],
        [ 1, 25, 13, 10, 14, 23,  1,  8],
        [10,  1, 21, 23, 20,  7, 17, 10]], device='cuda:4')
----
when input is [25] the target: 20
when input is [25, 20] the target: 1
when input is [25, 20, 1] the target: 9
when input is [25, 20, 1, 9] the target: 20
when input is [25, 20, 1, 9, 20] the target: 1
when input is [25, 20, 1, 9, 20, 1] the target: 28
when input is [25, 20, 1, 9, 20, 1, 28] the target: 13
when input is [25, 20, 1, 9, 20, 1, 28, 13] the target: 10
when input is [14] the target: 24
when input is [14, 24] the target: 25
when input is [14, 24, 25] the target: 14
when input is [14, 24, 25, 14] the target: 19
when input is [14, 24, 25, 14, 19] the ta

In [46]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size).to(device)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

# print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=800)[0].tolist()))


torch.Size([32, 32])
tensor(4.1553, device='cuda:4', grad_fn=<NllLossBackward0>)


In [47]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [48]:
train_loss = []
val_loss = []
batch_size = 32
for steps in range(10000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    train_loss.append(loss.item())
    xb_, yb_ = get_batch('val')
    logits, loss = m(xb_, yb_)
    print(f"Step {steps} Validation Loss: {loss.item()}")
    val_loss.append(loss.item())

# print(loss.item())


Step 0 Validation Loss: 4.042468070983887
Step 1 Validation Loss: 4.165435791015625
Step 2 Validation Loss: 4.212597370147705
Step 3 Validation Loss: 4.133847713470459
Step 4 Validation Loss: 4.082417964935303
Step 5 Validation Loss: 4.091870307922363
Step 6 Validation Loss: 4.236658096313477
Step 7 Validation Loss: 4.226165294647217
Step 8 Validation Loss: 4.100968360900879
Step 9 Validation Loss: 4.141510486602783
Step 10 Validation Loss: 4.169400691986084
Step 11 Validation Loss: 4.104821681976318
Step 12 Validation Loss: 4.173517227172852
Step 13 Validation Loss: 4.0626068115234375
Step 14 Validation Loss: 4.1319427490234375
Step 15 Validation Loss: 4.079352855682373
Step 16 Validation Loss: 4.0811896324157715
Step 17 Validation Loss: 4.13482666015625
Step 18 Validation Loss: 4.1955766677856445
Step 19 Validation Loss: 4.023454666137695
Step 20 Validation Loss: 4.132961273193359
Step 21 Validation Loss: 4.138492584228516
Step 22 Validation Loss: 4.06244421005249
Step 23 Validation 

In [50]:
# avg
train_loss = torch.tensor(train_loss)
val_loss = torch.tensor(val_loss)
print(f'Train Loss: {train_loss[-100:].mean()} +- {train_loss[-100:].std()}\n')
print(f'Validation Loss: {val_loss[-100:].mean()} +- {val_loss[-100:].std()}\n')

Train Loss: 2.3785154819488525 +- 0.06919090449810028

Validation Loss: 2.3936047554016113 +- 0.07320135086774826



/tmp/ipykernel_1694168/2449207959.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_loss = torch.tensor(train_loss)
/tmp/ipykernel_1694168/2449207959.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  val_loss = torch.tensor(val_loss)


In [51]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long, device=device), max_new_tokens=800)[0].tolist()))


gh, gokishaut couanthe msano r

oung. timai'dectongved ytemerst boueso s. put orint nces m eve'moneal ld t ayomanknt bam plu aseed..
toun an pi't. t cere'roharso lerobly warr ly fomalilln lea aisthe nche dimung onc haro myof whobutod wi the e hivethou?
d a scoortinco ciin. ige t pl, were ple wethentinu dbath me njuplldit?
i b.
t, dslin ther., a abs de itu don he theve.
it'sh whef t o is e.
lan he tt.
illattotwheme t zg ce. rbussi'seanth.
be
yodou bace lere ise no ruct mabeendo ybe t p.
y illlutht clled wxaspr.
cheso ayout howh clio fl ondibe re fore thowica avy f ove ir ating? liror ppllay ate, i'tel.
it b. t incck
llotoun gori cor cersinowwohindoury her ghi g ds t wi mot bughroryone ok l hanyot tyoryos edak, t aldfutwhe y, ange myon. camat journ'relioouptofongoouthllle fer c a cikerifo hm
